# 1. Model — Linear Regression

We combine each department's transactions into Sunday-to-Saturday weeks. A single
Linear Regression model learns four outputs: the totals for the next four weeks.
The inputs are the last four known weekly totals and the department category.

The CSV files do not say whether an absent record means no spending or missing
collection. We keep absent department-weeks as unknown, rather than setting them
to zero. We also conservatively exclude weeks that extend outside the file's
observed date range. This uses the dataset's coverage, not today's date.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from IPython.display import display

# Support starting Jupyter from either the notebook folder or project root.
data_folder = Path('../data')
if not (data_folder / 'forecast_data.csv').exists():
    data_folder = Path('ml/data')

expenses = pd.read_csv(data_folder / 'forecast_data.csv')
departments = pd.read_csv(data_folder / 'departments.csv')
expenses['expense_date'] = pd.to_datetime(expenses['expense_date'])

In [15]:
# A Sunday has dayofweek 6, so this gives its offset as zero.
expense_days = expenses['expense_date'].dt.normalize()
days_since_sunday = (expense_days.dt.dayofweek + 1) % 7
expenses['week_start'] = expense_days - pd.to_timedelta(days_since_sunday, unit='D')

first_day = expense_days.min()
last_day = expense_days.max()

first_week = expenses['week_start'].min()
if first_week < first_day:
    first_week = first_week + pd.Timedelta(weeks=1)

last_week = expenses['week_start'].max()
if last_week + pd.Timedelta(days=6) > last_day:
    last_week = last_week - pd.Timedelta(weeks=1)

complete_expenses = expenses[
    expenses['week_start'].between(first_week, last_week)
]
weekly = complete_expenses.groupby(
    ['week_start', 'department_id']
)['amount_sar'].sum().unstack('department_id')

In [17]:
# Keep every calendar week so a lag cannot accidentally skip a missing week.
week_dates = pd.date_range(first_week, last_week, freq='W-SUN')
weekly = weekly.reindex(index=week_dates, columns=departments['department_id'])
weekly.index.name = 'week_start'

missing_weeks = weekly.isna().sum()
print('Missing department-weeks:', int(missing_weeks.sum()))
print('Missing weeks remain unknown; samples needing them will be excluded.')
print('Weekly coverage:', first_week.date(), 'to', last_week.date())
print('Departments:', len(departments))

Missing department-weeks: 98
Missing weeks remain unknown; samples needing them will be excluded.
Weekly coverage: 2025-05-18 to 2026-06-21
Departments: 20


In [ ]:
# departments with the most missing weeks
print(missing_weeks.sort_values(ascending=False))

department_id
20    6
10    5
18    5
17    5
16    5
15    5
14    5
13    5
12    5
1     5
9     5
8     5
7     5
6     5
5     5
3     5
11    5
2     4
4     4
19    4
dtype: int64


In [ ]:
# Weeks with the highest number of missing sections
print(weekly.isna().sum(axis=1))

week_start
2025-05-18    20
2025-05-25    20
2025-06-01    20
2025-06-08    20
2025-06-15    17
2025-06-22     0
2025-06-29     0
2025-07-06     0
2025-07-13     0
2025-07-20     0
2025-07-27     0
2025-08-03     0
2025-08-10     0
2025-08-17     0
2025-08-24     0
2025-08-31     0
2025-09-07     0
2025-09-14     0
2025-09-21     0
2025-09-28     0
2025-10-05     0
2025-10-12     0
2025-10-19     0
2025-10-26     0
2025-11-02     0
2025-11-09     0
2025-11-16     0
2025-11-23     0
2025-11-30     0
2025-12-07     0
2025-12-14     0
2025-12-21     1
2025-12-28     0
2026-01-04     0
2026-01-11     0
2026-01-18     0
2026-01-25     0
2026-02-01     0
2026-02-08     0
2026-02-15     0
2026-02-22     0
2026-03-01     0
2026-03-08     0
2026-03-15     0
2026-03-22     0
2026-03-29     0
2026-04-05     0
2026-04-12     0
2026-04-19     0
2026-04-26     0
2026-05-03     0
2026-05-10     0
2026-05-17     0
2026-05-24     0
2026-05-31     0
2026-06-07     0
2026-06-14     0
2026-06-21     0
Fre

Each training row is dated by its first forecast week. `lag_1` is the week just
before that date, and `lag_4` is four weeks before it. The four targets begin at
that date. The forecast origin is the start date of the most recent completed
week; we make the forecast after that week ends.

In [23]:
sample_parts = []

for department_id in departments['department_id']:
    totals = weekly[department_id]
    department_samples = pd.DataFrame(index=weekly.index)
    department_samples['department_id'] = department_id

    for lag in range(1, 5):
        department_samples[f'lag_{lag}'] = totals.shift(lag)

    for horizon in range(1, 5):
        department_samples[f'target_{horizon}'] = totals.shift(-(horizon - 1))
    

    department_samples['origin_week'] = weekly.index - pd.Timedelta(weeks=1)
    department_samples['target_end_week'] = weekly.index + pd.Timedelta(weeks=3)
    sample_parts.append(department_samples)

samples = pd.concat(sample_parts).dropna().reset_index(drop=True)
lag_columns = ['lag_1', 'lag_2', 'lag_3', 'lag_4']
target_columns = ['target_1', 'target_2', 'target_3', 'target_4']
print('Usable training examples:', len(samples))

Usable training examples: 915


In [5]:
# Dummy columns treat department IDs as categories, not numeric quantities.
# The category list comes from department metadata, not future spending.
department_categories = departments['department_id'].tolist()
samples['department_id'] = pd.Categorical(
    samples['department_id'], categories=department_categories
)
X = pd.get_dummies(
    samples[lag_columns + ['department_id']],
    columns=['department_id'], drop_first=True, dtype=float
)
y = samples[target_columns]

# All these targets are within the completed historical weeks.
model = LinearRegression()
model.fit(X, y)

LinearRegression()

# 2. Horizon — Next 4 Weeks

Use each department's four most recent completed weekly totals to predict four
future totals directly. Predictions are in SAR. No actual future amounts are
used as inputs. Forecast dates follow the last completed week in this dataset.

In [6]:
latest_inputs = pd.DataFrame({'department_id': departments['department_id']})

for lag in range(1, 5):
    lag_week = last_week - pd.Timedelta(weeks=lag - 1)
    latest_inputs[f'lag_{lag}'] = latest_inputs['department_id'].map(weekly.loc[lag_week])

if latest_inputs[lag_columns].isna().any().any():
    raise ValueError('Each department needs four recent known weeks to forecast.')

latest_inputs['department_id'] = pd.Categorical(
    latest_inputs['department_id'], categories=department_categories
)
latest_X = pd.get_dummies(
    latest_inputs[lag_columns + ['department_id']],
    columns=['department_id'], drop_first=True, dtype=float
)
latest_X = latest_X.reindex(columns=X.columns, fill_value=0)
future_predictions = model.predict(latest_X)

In [7]:
forecast_table = departments[['department_id', 'name']].copy()
for horizon in range(1, 5):
    forecast_table[f'Week {horizon} (SAR)'] = future_predictions[:, horizon - 1]

forecast_dates = pd.DataFrame({
    'Horizon': ['Week 1', 'Week 2', 'Week 3', 'Week 4'],
    'Week starts (Sunday)': [
        last_week + pd.Timedelta(weeks=horizon) for horizon in range(1, 5)
    ]
})
display(forecast_dates)
display(forecast_table.round(2))

,Horizon,Week starts (Sunday)
0,Week 1,2026-06-28
1,Week 2,2026-07-05
2,Week 3,2026-07-12
3,Week 4,2026-07-19


,department_id,name,Week 1 (SAR),Week 2 (SAR),Week 3 (SAR),Week 4 (SAR)
0,1,Operations,1332121.42,1272020.17,1323753.14,1295143.25
1,2,Business Development,1269500.39,1391380.13,1321813.42,1295815.23
2,3,Research & Development,741970.11,753050.50,721177.93,708627.27
3,4,Facilities Management,600079.12,614807.45,654573.71,635771.44
4,5,Fleet & Logistics,915704.24,796861.80,687909.39,845633.21
5,6,Projects,261659.60,246354.46,216899.11,220662.93
6,7,Engineering,994485.12,1159252.16,1258827.26,1220243.57
7,8,Field Services,131617.69,144815.52,136598.76,112278.72
8,9,Hospitality & Events,76277.07,69291.46,81435.75,87030.84
9,10,Security,102611.79,117038.65,108611.18,115723.60


# 3. Baseline — Last-Week Baseline

The baseline repeats each department's last completed weekly total for all four
future weeks. This gives us a simple method to compare against Linear Regression.

In [8]:
last_week_values = latest_inputs['lag_1'].to_numpy()
baseline_predictions = np.repeat(last_week_values[:, None], 4, axis=1)

baseline_table = departments[['department_id', 'name']].copy()
for horizon in range(1, 5):
    baseline_table[f'Week {horizon} (SAR)'] = baseline_predictions[:, horizon - 1]

display(baseline_table.round(2))

,department_id,name,Week 1 (SAR),Week 2 (SAR),Week 3 (SAR),Week 4 (SAR)
0,1,Operations,1464812.92,1464812.92,1464812.92,1464812.92
1,2,Business Development,1071615.90,1071615.90,1071615.90,1071615.90
2,3,Research & Development,672084.47,672084.47,672084.47,672084.47
3,4,Facilities Management,863885.33,863885.33,863885.33,863885.33
4,5,Fleet & Logistics,811413.63,811413.63,811413.63,811413.63
5,6,Projects,286806.21,286806.21,286806.21,286806.21
6,7,Engineering,1059977.87,1059977.87,1059977.87,1059977.87
7,8,Field Services,132654.33,132654.33,132654.33,132654.33
8,9,Hospitality & Events,117282.91,117282.91,117282.91,117282.91
9,10,Security,96343.98,96343.98,96343.98,96343.98


# 4. Rolling Validation

We move the origin forward one week at a time. At each origin we train again,
using only examples whose entire four-week target is already known. Both methods
predict the same departments and future weeks. We use the final 12 eligible
origins, or all eligible origins if fewer exist. An origin is eligible when it
has training examples and all departments have four known input and target weeks.
Overlapping forecast windows are evaluated separately at each origin.

In [9]:
eligible_origins = []

for origin in sorted(samples['origin_week'].unique()):
    train_mask = samples['target_end_week'] <= origin
    validation_mask = samples['origin_week'] == origin
    validation_departments = samples.loc[validation_mask, 'department_id'].nunique()

    if train_mask.any() and validation_departments == len(departments):
        eligible_origins.append(origin)

validation_origins = eligible_origins[-12:]
if len(validation_origins) == 0:
    raise ValueError('Not enough complete weekly history for rolling validation.')

print('Rolling validation origins:', len(validation_origins))
print('First origin:', pd.Timestamp(validation_origins[0]).date())
print('Last origin:', pd.Timestamp(validation_origins[-1]).date())

Rolling validation origins: 12
First origin: 2026-03-08
Last origin: 2026-05-24


In [10]:
validation_parts = []

for origin in validation_origins:
    # A target ending after this origin is still in the future: exclude it.
    train_mask = samples['target_end_week'] <= origin
    validation_mask = samples['origin_week'] == origin

    rolling_model = LinearRegression()
    rolling_model.fit(X.loc[train_mask], y.loc[train_mask])
    regression_values = rolling_model.predict(X.loc[validation_mask])
    actual_values = y.loc[validation_mask].to_numpy()
    baseline_values = samples.loc[validation_mask, 'lag_1'].to_numpy()
    department_ids = samples.loc[validation_mask, 'department_id'].to_numpy()

    for horizon in range(1, 5):
        horizon_results = pd.DataFrame({
            'origin_week': origin,
            'department_id': department_ids,
            'horizon': horizon,
            'forecast_week': origin + pd.Timedelta(weeks=horizon),
            'actual': actual_values[:, horizon - 1],
            'baseline': baseline_values,
            'linear_regression': regression_values[:, horizon - 1]
        })
        validation_parts.append(horizon_results)

validation_results = pd.concat(validation_parts, ignore_index=True)
print('Evaluated department/week predictions:', len(validation_results))

Evaluated department/week predictions: 960


# 5. Numeric MAE Target

MAE is the average size of an error in SAR; smaller is better. Overall MAE treats
every department, forecast origin, and horizon equally. We also calculate MAE for
each of the four horizons. We keep full precision for calculations and round
only the displayed tables.

Our numeric target is 90% of the overall baseline MAE. Linear Regression achieves
the target if its overall MAE is at or below that value (at least 10% improvement
when baseline MAE is positive).

In [11]:
baseline_mae = mean_absolute_error(
    validation_results['actual'], validation_results['baseline']
)
regression_mae = mean_absolute_error(
    validation_results['actual'], validation_results['linear_regression']
)
mae_target = 0.90 * baseline_mae

comparison = pd.DataFrame({
    'Model': ['Last-Week Baseline', 'Linear Regression'],
    'MAE (SAR)': [baseline_mae, regression_mae],
    'Numeric MAE Target (SAR)': [mae_target, mae_target],
    'Target achieved': [baseline_mae <= mae_target, regression_mae <= mae_target]
})
display(comparison.round(2))

,Model,MAE (SAR),Numeric MAE Target (SAR),Target achieved
0,Last-Week Baseline,124677.26,112209.53,False
1,Linear Regression,115676.39,112209.53,False


In [12]:
horizon_rows = []

for horizon in range(1, 5):
    results = validation_results[validation_results['horizon'] == horizon]
    horizon_rows.append({
        'Horizon': f'Week {horizon}',
        'Last-Week Baseline MAE (SAR)': mean_absolute_error(
            results['actual'], results['baseline']
        ),
        'Linear Regression MAE (SAR)': mean_absolute_error(
            results['actual'], results['linear_regression']
        )
    })

horizon_comparison = pd.DataFrame(horizon_rows)
display(horizon_comparison.round(2))

,Horizon,Last-Week Baseline MAE (SAR),Linear Regression MAE (SAR)
0,Week 1,125086.56,115004.88
1,Week 2,128759.62,118093.99
2,Week 3,132047.65,114465.38
3,Week 4,112815.20,115141.32
